1. Thu thập và làm sạch dữ liệu

1.1. Thu thập dữ liệu từ vnstock

In [78]:
%pip install -U --extra-index-url https://vnstocks.com/api/simple vnstock

Looking in indexes: https://pypi.org/simple, https://vnstocks.com/api/simple
Note: you may need to restart the kernel to use updated packages.


In [79]:
import pandas as pd
import time
from pathlib import Path
from vnstock import Quote

# ==========================================
# PHẦN 0: KHAI BÁO THƯ MỤC LƯU DỮ LIỆU
# ==========================================

output_dir = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\unprocessed"
)

# Tạo thư mục nếu chưa tồn tại
output_dir.mkdir(parents=True, exist_ok=True)

# Đường dẫn lưu file
file_name = output_dir / "20tickers_raw.csv"
file_name_idx = output_dir / "vnindex_raw.csv"

# ==========================================
# PHẦN 1: KHAI BÁO THAM SỐ
# ==========================================

tickers = [
    # Nhóm Ngân hàng
    'VCB', 'CTG', 'BID', 'STB', 'MBB', 'TCB', 'VPB', 'HDB',

    # Nhóm Bất động sản
    'VIC', 'VHM', 'VRE', 'NVL', 'PDR',

    # Nhóm Năng lượng
    'GAS', 'PLX', 'PVD',

    # Nhóm Tiêu dùng - Bán lẻ
    'VNM', 'MWG', 'MSN', 'SAB'
]

start_date = '2018-09-01'
end_date = '2026-08-31'

all_data = []
failed_tickers = []

# ==========================================
# PHẦN 2: TẢI DỮ LIỆU 20 CỔ PHIẾU
# ==========================================

print("Bắt đầu tải dữ liệu 20 cổ phiếu...")

for tk in tickers:
    for attempt in range(3):
        try:
            q = Quote(symbol=tk, source='VCI')

            df = q.history(
                start=start_date,
                end=end_date,
                interval='1D'
            )

            if df is None or df.empty:
                raise ValueError("Không có dữ liệu được trả về.")

            df = df.copy()
            df['ticker'] = tk

            all_data.append(df)

            print(
                f"Thành công: {tk} - "
                f"Lấy được {len(df)} phiên."
            )

            break

        except Exception as e:
            print(f"Lỗi {tk} (lần {attempt + 1}/3): {e}")

            if attempt < 2:
                time.sleep(4)

    else:
        failed_tickers.append(tk)

    time.sleep(4)

# ==========================================
# PHẦN 3: XỬ LÝ VÀ LƯU DỮ LIỆU CỔ PHIẾU
# ==========================================

if all_data:
    df_all = pd.concat(all_data, ignore_index=True)

    df_all['time'] = pd.to_datetime(df_all['time'])
    df_all['ticker'] = df_all['ticker'].astype('category')

    for col in ['open', 'high', 'low', 'close', 'volume']:
        df_all[col] = pd.to_numeric(
            df_all[col], errors='coerce'
        )

    df_all = df_all.sort_values(
        ['ticker', 'time']
    ).reset_index(drop=True)

    # Lưu dữ liệu 20 cổ phiếu
    df_all.to_csv(
        file_name,
        index=False,
        encoding='utf-8-sig'
    )

    print("\nĐã lưu dữ liệu 20 cổ phiếu!")
    print(f"Đường dẫn: {file_name}")
    print(f"Tổng số dòng: {len(df_all):,}")
    print(f"Số mã tải thành công: {df_all['ticker'].nunique()}/20")

else:
    print("Không tải được dữ liệu của bất kỳ cổ phiếu nào.")

# ==========================================
# PHẦN 4: TẢI DỮ LIỆU VN-INDEX
# ==========================================

print("\n" + "-" * 50)
print("Bắt đầu tải dữ liệu VN-Index...")

try:
    q_index = Quote(symbol='VNINDEX', source='VCI')

    df_vnindex = q_index.history(
        start=start_date,
        end=end_date,
        interval='1D'
    )

    if df_vnindex is None or df_vnindex.empty:
        raise ValueError("Không có dữ liệu VN-Index được trả về.")

    df_vnindex = df_vnindex.copy()

    df_vnindex['ticker'] = 'VNINDEX'
    df_vnindex['time'] = pd.to_datetime(df_vnindex['time'])

    for col in ['open', 'high', 'low', 'close', 'volume']:
        df_vnindex[col] = pd.to_numeric(
            df_vnindex[col], errors='coerce'
        )

    # Giữ dữ liệu từ ngày 30/08/2018 nếu nguồn có trả về
    df_vnindex = df_vnindex[
        df_vnindex['time'] >= pd.Timestamp('2018-08-30')
    ].copy()

    # Sắp xếp và chọn các cột cần thiết
    df_vnindex = df_vnindex[
        ['time', 'ticker', 'open', 'close', 'high', 'low', 'volume']
    ]

    df_vnindex = df_vnindex.sort_values(
        'time'
    ).reset_index(drop=True)

    # Lưu dữ liệu VN-Index
    df_vnindex.to_csv(
        file_name_idx,
        index=False,
        encoding='utf-8-sig'
    )

    print("\nTải VN-Index thành công!")
    print(f"Đường dẫn: {file_name_idx}")
    print(f"Tổng số phiên: {len(df_vnindex):,}")

    print(
        f"Ngày bắt đầu: "
        f"{df_vnindex['time'].min().strftime('%Y-%m-%d')}"
    )

    print(
        f"Ngày kết thúc: "
        f"{df_vnindex['time'].max().strftime('%Y-%m-%d')}"
    )

except Exception as e:
    print(f"Lỗi khi tải dữ liệu VN-Index: {e}")

# ==========================================
# PHẦN 5: BÁO CÁO KẾT QUẢ
# ==========================================

print("\n" + "=" * 50)
print("BÁO CÁO TẢI DỮ LIỆU")
print("=" * 50)

if all_data:
    print(f"File cổ phiếu: {file_name}")

if file_name_idx.exists():
    print(f"File VN-Index: {file_name_idx}")

if failed_tickers:
    print(f"Các mã tải thất bại: {failed_tickers}")
else:
    print("Tất cả 20 mã cổ phiếu đều tải thành công.")

Bắt đầu tải dữ liệu 20 cổ phiếu...
Thành công: VCB - Lấy được 1997 phiên.
Thành công: CTG - Lấy được 1997 phiên.
Thành công: BID - Lấy được 1997 phiên.
Thành công: STB - Lấy được 1997 phiên.
Thành công: MBB - Lấy được 1997 phiên.
Thành công: TCB - Lấy được 1997 phiên.
Thành công: VPB - Lấy được 1997 phiên.
Thành công: HDB - Lấy được 1997 phiên.
Thành công: VIC - Lấy được 1997 phiên.
Thành công: VHM - Lấy được 1997 phiên.
Thành công: VRE - Lấy được 1997 phiên.
Thành công: NVL - Lấy được 1997 phiên.
Thành công: PDR - Lấy được 1997 phiên.
Thành công: GAS - Lấy được 1997 phiên.
Thành công: PLX - Lấy được 1997 phiên.
Thành công: PVD - Lấy được 1997 phiên.
Thành công: VNM - Lấy được 1997 phiên.
Thành công: MWG - Lấy được 1997 phiên.
Thành công: MSN - Lấy được 1997 phiên.
Thành công: SAB - Lấy được 1997 phiên.

Đã lưu dữ liệu 20 cổ phiếu!
Đường dẫn: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\unprocessed\20tickers_raw.csv
Tổng số dòng: 39,

In [80]:
# KIỂM TRA NGÀY GIAO DỊCH TỪNG TICKER

if all_data:
    start_check = pd.Timestamp('2018-08-30')
    end_check = pd.Timestamp('2026-08-28')

    date_check = (
        df_all.groupby('ticker', observed=True)['time']
        .agg(
            min_date='min',
            max_date='max'
        )
        .reset_index()
    )

    date_check['min_date_valid'] = date_check['min_date'].between(
        start_check, end_check
    )

    date_check['max_date_valid'] = date_check['max_date'].between(
        start_check, end_check
    )

    date_check['result'] = (
        date_check['min_date_valid']
        & date_check['max_date_valid']
    ).map({
        True: 'ĐẠT',
        False: 'KHÔNG ĐẠT'
    })

    print("\nKIỂM TRA NGÀY GIAO DỊCH TỪNG TICKER")
    print("Khoảng yêu cầu: 2018-08-30 đến 2026-08-28\n")

    print(
        date_check.to_string(
            index=False,
            formatters={
                'min_date': lambda x: x.strftime('%Y-%m-%d'),
                'max_date': lambda x: x.strftime('%Y-%m-%d')
            }
        )
    )

    print("\nSố ticker đạt:",
          (date_check['result'] == 'ĐẠT').sum())

    print("Số ticker không đạt:",
          (date_check['result'] == 'KHÔNG ĐẠT').sum())


KIỂM TRA NGÀY GIAO DỊCH TỪNG TICKER
Khoảng yêu cầu: 2018-08-30 đến 2026-08-28

ticker   min_date   max_date  min_date_valid  max_date_valid result
   BID 2018-08-30 2026-08-28            True            True    ĐẠT
   CTG 2018-08-30 2026-08-28            True            True    ĐẠT
   GAS 2018-08-30 2026-08-28            True            True    ĐẠT
   HDB 2018-08-30 2026-08-28            True            True    ĐẠT
   MBB 2018-08-30 2026-08-28            True            True    ĐẠT
   MSN 2018-08-30 2026-08-28            True            True    ĐẠT
   MWG 2018-08-30 2026-08-28            True            True    ĐẠT
   NVL 2018-08-30 2026-08-28            True            True    ĐẠT
   PDR 2018-08-30 2026-08-28            True            True    ĐẠT
   PLX 2018-08-30 2026-08-28            True            True    ĐẠT
   PVD 2018-08-30 2026-08-28            True            True    ĐẠT
   SAB 2018-08-30 2026-08-28            True            True    ĐẠT
   STB 2018-08-30 2026-08-28        

In [81]:
# KIỂM TRA NGÀY GIAO DỊCH VN-INDEX

if 'df_vnindex' in locals() and not df_vnindex.empty:
    start_check = pd.Timestamp('2018-08-30')
    end_check = pd.Timestamp('2026-08-28')

    min_date = df_vnindex['time'].min()
    max_date = df_vnindex['time'].max()

    min_valid = start_check <= min_date <= end_check
    max_valid = start_check <= max_date <= end_check

    print("\nKIỂM TRA NGÀY GIAO DỊCH VN-INDEX")
    print(f"Ngày giao dịch nhỏ nhất: {min_date:%Y-%m-%d}")
    print(f"Ngày giao dịch lớn nhất: {max_date:%Y-%m-%d}")
    print(f"Ngày nhỏ nhất nằm trong khoảng: {'ĐẠT' if min_valid else 'KHÔNG ĐẠT'}")
    print(f"Ngày lớn nhất nằm trong khoảng: {'ĐẠT' if max_valid else 'KHÔNG ĐẠT'}")
    print(
        f"Kết quả: {'ĐẠT' if min_valid and max_valid else 'KHÔNG ĐẠT'}"
    )
else:
    print("\nKhông có dữ liệu VN-Index để kiểm tra.")


KIỂM TRA NGÀY GIAO DỊCH VN-INDEX
Ngày giao dịch nhỏ nhất: 2018-08-30
Ngày giao dịch lớn nhất: 2026-08-28
Ngày nhỏ nhất nằm trong khoảng: ĐẠT
Ngày lớn nhất nằm trong khoảng: ĐẠT
Kết quả: ĐẠT


1.2. Chuẩn hóa kiểu dữ liệu & kiểm tra tính đầy đủ

In [82]:
from pathlib import Path

drive_path = Path("G:/My Drive")

for name in ["20tickers_raw.csv", "vnindex_raw.csv"]:
    print(f"\n===== Tìm {name} =====")
    files = list(drive_path.rglob(name))

    if files:
        for f in files:
            print("Đường dẫn:", f)
    else:
        print("Không tìm thấy file.")


===== Tìm 20tickers_raw.csv =====
Đường dẫn: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\unprocessed\20tickers_raw.csv

===== Tìm vnindex_raw.csv =====
Đường dẫn: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\unprocessed\vnindex_raw.csv


In [83]:
import pandas as pd
from pathlib import Path
raw_path = Path.home() / "Downloads"
#Đọc dữ liệu từ file CSV
tickers = pd.read_csv(r"\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\unprocessed\20tickers_raw.csv")
vnindex = pd.read_csv(r"\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\unprocessed\vnindex_raw.csv")

print("Đọc dữ liệu thành công!")

# Chuyển kiểu dữ liệu
tickers["time"] = pd.to_datetime(tickers["time"])
vnindex["time"] = pd.to_datetime(vnindex["time"])

tickers["ticker"] = tickers["ticker"].astype("category")
vnindex["ticker"] = vnindex["ticker"].astype("category")

# Tập ngày giao dịch chuẩn của VNINDEX
vnindex_days = set(vnindex["time"].dt.date.unique())

print("\nSố phiên VNINDEX:", len(vnindex_days))

# Kiểm tra số phiên và ngày thiếu của từng cổ phiếu
for ticker in tickers["ticker"].cat.categories:
    ticker_df = tickers[tickers["ticker"] == ticker]

    ticker_days = set(ticker_df["time"].dt.date.unique())

    missing_days = sorted(vnindex_days - ticker_days)

    print("\n====================")
    print("Ticker:", ticker)
    print("Số phiên:", len(ticker_days))
    print("Thiếu:", len(missing_days))

    if missing_days:
        print("Ngày thiếu:")
        for d in missing_days:
            print(d)

#  LƯU FILE ĐÃ CHUYỂN KIỂU DỮ LIỆU

# Đường dẫn file đầu ra
tickers_output = (
    Path(r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu")
    / "data"
    / "processed"
    / "preparation_preprocessing"
    / "20tickers_typed.csv"
)
vnindex_output = (
    Path(r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu")
    / "data"
    / "processed"
    / "preparation_preprocessing"
    / "vnindex_typed.csv"
)

# Lưu dữ liệu
tickers.to_csv(
    tickers_output,
    index=False,
    encoding="utf-8-sig"
)

vnindex.to_csv(
    vnindex_output,
    index=False,
    encoding="utf-8-sig"
)

# Thông báo kết quả
print("\n===== LƯU FILE THÀNH CÔNG =====")
print("File cổ phiếu:", tickers_output)
print("File VN-Index:", vnindex_output)

Đọc dữ liệu thành công!

Số phiên VNINDEX: 1997

Ticker: BID
Số phiên: 1997
Thiếu: 0

Ticker: CTG
Số phiên: 1997
Thiếu: 0

Ticker: GAS
Số phiên: 1997
Thiếu: 0

Ticker: HDB
Số phiên: 1997
Thiếu: 0

Ticker: MBB
Số phiên: 1997
Thiếu: 0

Ticker: MSN
Số phiên: 1997
Thiếu: 0

Ticker: MWG
Số phiên: 1997
Thiếu: 0

Ticker: NVL
Số phiên: 1997
Thiếu: 0

Ticker: PDR
Số phiên: 1997
Thiếu: 0

Ticker: PLX
Số phiên: 1997
Thiếu: 0

Ticker: PVD
Số phiên: 1997
Thiếu: 0

Ticker: SAB
Số phiên: 1997
Thiếu: 0

Ticker: STB
Số phiên: 1997
Thiếu: 0

Ticker: TCB
Số phiên: 1997
Thiếu: 0

Ticker: VCB
Số phiên: 1997
Thiếu: 0

Ticker: VHM
Số phiên: 1997
Thiếu: 0

Ticker: VIC
Số phiên: 1997
Thiếu: 0

Ticker: VNM
Số phiên: 1997
Thiếu: 0

Ticker: VPB
Số phiên: 1997
Thiếu: 0

Ticker: VRE
Số phiên: 1997
Thiếu: 0

===== LƯU FILE THÀNH CÔNG =====
File cổ phiếu: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\20tickers_typed.csv
File VN-In

1.3. Kiểm tra chất lượng dữ liệu

In [84]:
# Đọc dữ liệu
tickers = pd.read_csv(r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\20tickers_typed.csv")
vnindex = pd.read_csv(r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\vnindex_typed.csv")


print("Tickers:", tickers.shape)
print("VN-Index:", vnindex.shape)


# CHUẨN HÓA KIỂU DỮ LIỆU

for df in [tickers, vnindex]:
    df["time"] = pd.to_datetime(df["time"])
    for col in ["open", "close", "high", "low", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        
tickers = tickers.sort_values(["ticker", "time"]).reset_index(drop=True)
vnindex = vnindex.sort_values("time").reset_index(drop=True)

Tickers: (39940, 7)
VN-Index: (1997, 7)


In [85]:
# KIỂM TRA LOGIC GIÁ
# Low <= Open <= High
# Low <= Close <= High

print("=" * 70)
print("1. KIỂM TRA LOGIC GIÁ")
print("=" * 70)


def check_price_logic(df):
    errors = []
    for idx, row in df.iterrows():
        low = row["low"]
        open_price = row["open"]
        high = row["high"]
        close = row["close"]


        if pd.isna(low) or pd.isna(open_price) or pd.isna(high) or pd.isna(close):
            continue


        # Low > Open
        if low > open_price:
            errors.append({
                "ticker": row["ticker"],
                "time": row["time"],
                "error": "Low > Open",
                "open": open_price,
                "close": close,
                "high": high,
                "low": low
            })


        # Open > High
        if open_price > high:
            errors.append({
                "ticker": row["ticker"],
                "time": row["time"],
                "error": "Open > High",
                "open": open_price,
                "close": close,
                "high": high,
                "low": low
            })


        # Low > Close
        if low > close:
            errors.append({
                "ticker": row["ticker"],
                "time": row["time"],
                "error": "Low > Close",
                "open": open_price,
                "close": close,
                "high": high,
                "low": low
            })


        # Close > High
        if close > high:
            errors.append({
                "ticker": row["ticker"],
                "time": row["time"],
                "error": "Close > High",
                "open": open_price,
                "close": close,
                "high": high,
                "low": low
            })


    return pd.DataFrame(errors)


price_errors_tickers = check_price_logic(tickers)
price_errors_vnindex = check_price_logic(vnindex)


price_errors = pd.concat([price_errors_tickers, price_errors_vnindex], ignore_index=True)


print("Số dòng vi phạm logic giá:", len(price_errors))
if len(price_errors) == 0:
    print("Không phát hiện dòng nào vi phạm logic giá.")
else:
    print(price_errors.to_string(index=False))

1. KIỂM TRA LOGIC GIÁ
Số dòng vi phạm logic giá: 4
 ticker       time       error    open   close    high     low
VNINDEX 2019-06-24  Low > Open  949.48  962.85  967.11  962.63
VNINDEX 2019-06-25  Low > Open  949.48  960.13  963.23  960.06
VNINDEX 2019-06-26  Low > Open  949.48  959.13  962.70  958.49
VNINDEX 2021-08-23 Open > High 1329.43 1298.86 1326.07 1298.86


In [86]:
# KIỂM TRA DÒNG TRÙNG LẶP 100%

print("\n" + "=" * 70)
print("2. KIỂM TRA DÒNG TRÙNG LẶP 100%")
print("=" * 70)


def check_full_duplicates(df):
    duplicate_columns = ["time", "ticker", "open", "close", "high", "low", "volume"]
    duplicates = df[df.duplicated(subset=duplicate_columns, keep=False)].copy()
    return duplicates


duplicate_tickers = check_full_duplicates(tickers)
duplicate_vnindex = check_full_duplicates(vnindex)


duplicate_full = pd.concat([duplicate_tickers, duplicate_vnindex], ignore_index=True)


print("Số dòng trùng 100%:", len(duplicate_full))
if len(duplicate_full) == 0:
    print("Không phát hiện dòng trùng 100%.")


2. KIỂM TRA DÒNG TRÙNG LẶP 100%
Số dòng trùng 100%: 0
Không phát hiện dòng trùng 100%.


In [87]:
# KIỂM TRA TRÙNG TICKER + TIME

print("\n" + "=" * 70)
print("3. KIỂM TRA TRÙNG TICKER + TIME")
print("=" * 70)


duplicate_key = tickers[tickers.duplicated(subset=["ticker", "time"], keep=False)].copy()


print("Số dòng trùng ticker + time:", len(duplicate_key))
if len(duplicate_key) == 0:
    print("Không phát hiện trường hợp trùng ticker + time.")
else:
    print("\nCác trường hợp cần kiểm tra:")
    print(
        duplicate_key[
            ["time", "ticker", "open", "close", "high", "low", "volume"]
        ].to_string(index=False)
    )


3. KIỂM TRA TRÙNG TICKER + TIME
Số dòng trùng ticker + time: 0
Không phát hiện trường hợp trùng ticker + time.


In [88]:
# TÍNH RETURN CHO 20 MÃ
# Return = (Close_t - Close_(t-1)) / Close_(t-1)
# (Kiểm tra ở mục 2 và 3 không có dòng trùng nên không cần bước loại trùng)
# ============================================================
print("\n" + "=" * 70)
print("4. TÍNH RETURN - 20 MÃ")
print("=" * 70)


tickers = tickers.sort_values(["ticker", "time"]).reset_index(drop=True)
tickers["Return"] = tickers.groupby("ticker")["close"].pct_change()


# ============================================================
# TÍNH RETURN CHO VNINDEX
# ============================================================
vnindex = vnindex.sort_values("time").reset_index(drop=True)
vnindex["Return"] = vnindex["close"].pct_change()


# ============================================================
# LỌC RETURN > +/-7% - 20 MÃ
# ============================================================
print("\n" + "=" * 70)
print("5. KIỂM TRA RETURN > +/-7% - 20 MÃ")
print("=" * 70)


abnormal_tickers = tickers[tickers["Return"].abs() > 0.7].copy()
abnormal_tickers["Return_percent"] = abnormal_tickers["Return"] * 100


print("Số trường hợp Return > +/-7%:", len(abnormal_tickers))
if len(abnormal_tickers) > 0:
    print(
        abnormal_tickers[
            ["time", "ticker", "close", "Return_percent"]
        ].to_string(index=False)
    )
else:
    print("Không phát hiện ngày nào có Return vượt +/-7%.")


# ============================================================
# LỌC RETURN > +/-7% - VNINDEX
# ============================================================
print("\n" + "=" * 70)
print("6. KIỂM TRA RETURN > +/-7% - VNINDEX")
print("=" * 70)


abnormal_vnindex = vnindex[vnindex["Return"].abs() > 0.7].copy()
abnormal_vnindex["Return_percent"] = abnormal_vnindex["Return"] * 100


print("Số trường hợp Return > +/-7%:", len(abnormal_vnindex))
if len(abnormal_vnindex) > 0:
    print(
        abnormal_vnindex[
            ["time", "ticker", "close", "Return_percent"]
        ].to_string(index=False)
    )
else:
    print("Không phát hiện ngày nào có Return vượt +/-7%.")


4. TÍNH RETURN - 20 MÃ

5. KIỂM TRA RETURN > +/-7% - 20 MÃ
Số trường hợp Return > +/-7%: 0
Không phát hiện ngày nào có Return vượt +/-7%.

6. KIỂM TRA RETURN > +/-7% - VNINDEX
Số trường hợp Return > +/-7%: 0
Không phát hiện ngày nào có Return vượt +/-7%.


In [89]:
# KIỂM TRA ĐIỀU CHỈNH GIÁ DO CHIA TÁCH / CỔ TỨC (3 MÃ)

print("\n" + "=" * 70)
print("7. ĐỐI CHIẾU ĐIỀU CHỈNH GIÁ DO CHIA TÁCH / CỔ TỨC (3 MÃ)")
print("=" * 70)


corporate_events = [
    {
        "ticker": "MBB",
        "date": "2021-07-12",
        "event": "Trả cổ tức bằng cổ phiếu tỷ lệ 35%",
    },
    {
        "ticker": "HDB",
        "date": "2021-08-26",
        "event": "Trả cổ tức bằng cổ phiếu tỷ lệ 25%",
    },
    {
        "ticker": "VNM",
        "date": "2020-09-29",
        "event": "Cổ phiếu thưởng tỷ lệ 20% & Cổ tức tiền mặt 2.000đ/cp",
    },
]


event_rows = []


for ev in corporate_events:
    tk = ev["ticker"]
    event_date = pd.to_datetime(ev["date"])


    sub_df = tickers[tickers["ticker"] == tk].copy()


    # Lấy cửa sổ quanh ngày GDKHQ (+/- 5 ngày theo lịch)
    window = sub_df[
        (sub_df["time"] >= event_date - pd.Timedelta(days=5))
        & (sub_df["time"] <= event_date + pd.Timedelta(days=5))
    ]


    print(f"\n--- {tk}: {ev['event']} (Ngày GDKHQ: {ev['date']}) ---")
    print(window[["time", "ticker", "close", "Return"]].to_string(index=False))


    for _, r in window.iterrows():
        event_rows.append({
            "ticker": tk,
            "event": ev["event"],
            "event_date": ev["date"],
            "time": r["time"],
            "close": r["close"],
            "return": r["Return"],
        })


7. ĐỐI CHIẾU ĐIỀU CHỈNH GIÁ DO CHIA TÁCH / CỔ TỨC (3 MÃ)

--- MBB: Trả cổ tức bằng cổ phiếu tỷ lệ 35% (Ngày GDKHQ: 2021-07-12) ---
      time ticker  close    Return
2021-07-07    MBB  11.27  0.049348
2021-07-08    MBB  11.27  0.000000
2021-07-09    MBB  11.21 -0.005324
2021-07-12    MBB  10.77 -0.039251
2021-07-13    MBB  10.77  0.000000
2021-07-14    MBB  10.43 -0.031569
2021-07-15    MBB  10.59  0.015340
2021-07-16    MBB  10.52 -0.006610

--- HDB: Trả cổ tức bằng cổ phiếu tỷ lệ 25% (Ngày GDKHQ: 2021-08-26) ---
      time ticker  close    Return
2021-08-23    HDB   8.42 -0.026590
2021-08-24    HDB   8.45  0.003563
2021-08-25    HDB   8.47  0.002367
2021-08-26    HDB   8.41 -0.007084
2021-08-27    HDB   8.30 -0.013080
2021-08-30    HDB   8.37  0.008434
2021-08-31    HDB   8.13 -0.028674

--- VNM: Cổ phiếu thưởng tỷ lệ 20% & Cổ tức tiền mặt 2.000đ/cp (Ngày GDKHQ: 2020-09-29) ---
      time ticker  close    Return
2020-09-24    VNM  76.21 -0.014738
2020-09-25    VNM  76.45  0.003149
2

In [90]:
# LƯU TẤT CẢ FILE OUTPUT

from pathlib import Path
import pandas as pd

# Thư mục lưu output
output_path = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing"
)

# Tạo thư mục nếu chưa tồn tại
output_path.mkdir(parents=True, exist_ok=True)

# Dữ liệu có thêm cột Return
tickers.to_csv(output_path / "20tickers_return.csv", index=False)
vnindex.to_csv(output_path / "vnindex_return.csv", index=False)

pd.DataFrame(event_rows).to_csv(
    output_path / "corporate_actions_check.csv",
    index=False
)

print("\n" + "=" * 70)
print("HOÀN THÀNH LƯU CÁC FILE OUTPUT")
print("=" * 70)
print("Thư mục lưu:", output_path)

# Kiểm tra danh sách file đã lưu
for file in sorted(output_path.glob("*.csv")):
    print(file.name)


HOÀN THÀNH LƯU CÁC FILE OUTPUT
Thư mục lưu: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing
20tickers_return.csv
20tickers_typed.csv
corporate_actions_check.csv
vnindex_return.csv
vnindex_typed.csv


2. Xác định giai đoạn khủng hoảng

2.1. Tính chuỗi Drawdown của VN-Index

In [91]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)

input_path = (
    PROJECT_DIR / "data" / "processed"
    / "preparation_preprocessing" / "vnindex_return.csv"
)

# Đọc dữ liệu
df = pd.read_csv(input_path)

# Kiểm tra tên cột
print("Các cột trong file:", df.columns.tolist())

# Chuẩn hóa tên cột
df.columns = df.columns.str.strip().str.lower()

# Nếu cột thời gian có tên khác, đổi về time
if "time" not in df.columns:
    for col in ["date", "trading_date", "timestamp"]:
        if col in df.columns:
            df = df.rename(columns={col: "time"})
            break

# Kiểm tra cột bắt buộc
if "time" not in df.columns:
    raise ValueError("Không tìm thấy cột thời gian. Hãy kiểm tra tên cột trong file CSV.")

if "close" not in df.columns:
    raise ValueError("Không tìm thấy cột close trong file CSV.")

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

# Tính toán Peak (cummax) và Drawdown theo công thức
df["peak"] = df["close"].cummax()
df["drawdown"] = (df["close"] - df["peak"]) / df["peak"]


# Xuất file dữ liệu đã bổ sung cột mới
output_csv = r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\vnindex_drawdown.csv"
df.to_csv(output_csv, index=False)
print(f"Đã lưu file dữ liệu mới tại: {output_csv}")


# Vẽ biểu đồ Drawdown theo thời gian
plt.figure(figsize=(12, 6))
plt.plot(
    df["time"],
    df["drawdown"] * 100,
    color="#d9534f",
    linewidth=1.2,
    label="VN-Index Drawdown",
)


plt.title(
    "Chuỗi Drawdown VN-Index Theo Thời Gian", fontsize=14, fontweight="bold"
)
plt.xlabel("Thời gian")
plt.ylabel("Drawdown (%)")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower left")
plt.tight_layout()


# Lưu file biểu đồ
output_img = r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\reports\figures\preparation\vnindex_drawdown.png"
plt.savefig(output_img, dpi=300)
plt.close()
print(f"Đã xuất biểu đồ tại: {output_img}")

Các cột trong file: ['time', 'ticker', 'open', 'close', 'high', 'low', 'volume', 'Return']
Đã lưu file dữ liệu mới tại: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\vnindex_drawdown.csv
Đã xuất biểu đồ tại: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\reports\figures\preparation\vnindex_drawdown.png


In [92]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Đảm bảo thư mục lưu trữ tồn tại
figures_dir = r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\reports\figures\preparation"
os.makedirs(figures_dir, exist_ok=True)

# Đọc dữ liệu drawdown đã tính ở bước trước
input_path = r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\vnindex_drawdown.csv"
df = pd.read_csv(input_path)
df["time"] = pd.to_datetime(df["time"])


# Tính toán các mức phân vị (Percentiles)
p5 = df["drawdown"].quantile(0.05) * 100
p10 = df["drawdown"].quantile(0.10) * 100


print("=== THỐNG KÊ PHÂN VỊ DRAWDOWN ===")
print(f"Phân vị 5%  (5% số phiên giảm sâu nhất): {p5:.2f}%")
print(f"Phân vị 10% (10% số phiên giảm sâu nhất): {p10:.2f}%\n")


# Vẽ biểu đồ Histogram phân phối Drawdown
plt.figure(figsize=(10, 6))
plt.hist(
    df["drawdown"] * 100,
    bins=50,
    color="#2b5c8f",
    edgecolor="black",
    alpha=0.75,
)


# Thêm các đường ranh giới phân vị và ngưỡng đề xuất
plt.axvline(
    x=-15,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Ngưỡng đề xuất chính thức (-15%)",
)
plt.axvline(
    x=p5,
    color="orange",
    linestyle=":",
    linewidth=1.5,
    label=f"Phân vị 5% ({p5:.1f}%)",
)
plt.axvline(
    x=p10,
    color="green",
    linestyle=":",
    linewidth=1.5,
    label=f"Phân vị 10% ({p10:.1f}%)",
)


plt.title("Phân Phối Tần Suất Drawdown VN-Index", fontsize=14, fontweight="bold")
plt.xlabel("Drawdown (%)")
plt.ylabel("Tần suất (Số phiên)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()


# Lưu biểu đồ histogram
hist_output_path = os.path.join(figures_dir, "vnindex_histogram.png")
plt.savefig(hist_output_path, dpi=300)
plt.close()
print(f"Đã xuất biểu đồ tại: {hist_output_path}\n")




# Hàm đếm số đợt sụt giảm vượt ngưỡng
def count_crisis_periods(data, threshold_pct):
    threshold = threshold_pct / 100.0
    in_crisis = False
    count = 0


    for dd in data["drawdown"]:
        if not in_crisis and dd < threshold:
            in_crisis = True
            count += 1
        elif in_crisis and dd >= threshold:
            in_crisis = False


    return count




# Thử nghiệm quét qua các mức ngưỡng lân cận
test_thresholds = [-10, -12, -15, -18, -20]
print("=== KẾT QUẢ THỬ NGHIỆM CÁC NGƯỠNG ===")
for th in test_thresholds:
    periods = count_crisis_periods(df, th)
    print(f"Ngưỡng {th}% : Tìm thấy {periods} giai đoạn")

=== THỐNG KÊ PHÂN VỊ DRAWDOWN ===
Phân vị 5%  (5% số phiên giảm sâu nhất): -31.13%
Phân vị 10% (10% số phiên giảm sâu nhất): -27.89%

Đã xuất biểu đồ tại: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\reports\figures\preparation\vnindex_histogram.png

=== KẾT QUẢ THỬ NGHIỆM CÁC NGƯỠNG ===
Ngưỡng -10% : Tìm thấy 17 giai đoạn
Ngưỡng -12% : Tìm thấy 16 giai đoạn
Ngưỡng -15% : Tìm thấy 13 giai đoạn
Ngưỡng -18% : Tìm thấy 24 giai đoạn
Ngưỡng -20% : Tìm thấy 18 giai đoạn


2.2. Trích xuất danh sách các giai đoạn khủng hoảng

In [93]:
import os
import pandas as pd


# Khởi tạo thư mục chứa đầu ra
os.makedirs("data/processed", exist_ok=True)


# Đọc dữ liệu Drawdown VN-Index
input_path = r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\vnindex_drawdown.csv"
df = pd.read_csv(input_path)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)




# Quét tự động các đợt khủng hoảng với ngưỡng -15%
def extract_crisis_periods(data, threshold=-0.15):
    in_crisis = False
    periods = []
    start_date = None
    max_dd = 0.0


    for idx, row in data.iterrows():
        dd = row["drawdown"]
        current_date = row["time"]


        if not in_crisis and dd < threshold:
            in_crisis = True
            start_date = current_date
            max_dd = dd
        elif in_crisis:
            if dd < max_dd:
                max_dd = dd
            if dd >= threshold:
                in_crisis = False
                periods.append(
                    {
                        "start_date": start_date.strftime("%Y-%m-%d"),
                        "end_date": current_date.strftime("%Y-%m-%d"),
                        "max_drawdown": f"{max_dd * 100:.2f}%",
                    }
                )


    if in_crisis:
        periods.append(
            {
                "start_date": start_date.strftime("%Y-%m-%d"),
                "end_date": data.iloc[-1]["time"].strftime("%Y-%m-%d"),
                "max_drawdown": f"{max_dd * 100:.2f}%",
            }
        )


    return pd.DataFrame(periods)




crisis_df = extract_crisis_periods(df, threshold=-0.15)


# Danh mục tin tức tra cứu thực tế theo đúng start_date (Bạn có thể tự cập nhật/thay link)
news_mapping = {
    "2018-04-19": {
        "stage_name": "Chiến tranh thương mại Mỹ - Trung & FED nâng lãi suất",
        "news_url": "https://vietstock.vn/...",  # Dán link bài báo bạn tìm được vào đây
    },
    "2020-02-03": {
        "stage_name": "Bùng phát đại dịch COVID-19 toàn cầu",
        "news_url": "https://vnexpress.net/...",
    },
    "2022-04-18": {
        "stage_name": "Xử lý sai phạm Tân Hoàng Minh, Vạn Thịnh Phát & Sự cố SCB",
        "news_url": "https://cafef.vn/...",
    },
    "2023-09-25": {
        "stage_name": "Áp lực Tỷ giá USD/VND tăng cao & Ngân hàng Nhà nước hút tiền qua tín phiếu",
        "news_url": "https://vneconomy.vn/...",
    },
}


# Gán tên sự kiện và link bài báo dựa trên start_date chuẩn xác
crisis_df["stage_name"] = crisis_df["start_date"].apply(
    lambda x: news_mapping.get(x, {}).get(
        "stage_name", "Điều chỉnh vĩ mô sâu"
    )
)
crisis_df["news_url"] = crisis_df["start_date"].apply(
    lambda x: news_mapping.get(x, {}).get("news_url", "Cần bổ sung link")
)

# Xuất file CSV hoàn chỉnh
output_csv = r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\crisis_periods.csv"

crisis_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Đã lưu file tại: {output_csv}")
print(crisis_df)

Đã lưu file tại: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\crisis_periods.csv
    start_date    end_date max_drawdown            stage_name  \
0   2020-03-09  2020-06-01      -35.68%  Điều chỉnh vĩ mô sâu   
1   2020-06-11  2020-06-22      -18.78%  Điều chỉnh vĩ mô sâu   
2   2020-06-23  2020-07-09      -19.49%  Điều chỉnh vĩ mô sâu   
3   2020-07-13  2020-07-16      -15.30%  Điều chỉnh vĩ mô sâu   
4   2020-07-20  2020-08-25      -23.39%  Điều chỉnh vĩ mô sâu   
5   2022-05-09  2022-05-11      -16.94%  Điều chỉnh vĩ mô sâu   
6   2022-05-12  2022-06-01      -23.33%  Điều chỉnh vĩ mô sâu   
7   2022-06-02  2022-06-08      -15.74%  Điều chỉnh vĩ mô sâu   
8   2022-06-10  2024-06-12      -40.34%  Điều chỉnh vĩ mô sâu   
9   2024-06-14  2025-02-24      -22.28%  Điều chỉnh vĩ mô sâu   
10  2025-04-03  2025-05-14      -28.41%  Điều chỉnh vĩ mô sâu   
11  2025-05-19  2025-05-20      -15.20%  Điều chỉn

In [94]:
#4-mới Từ điển ánh xạ chính xác theo ngày start_date
news_mapping = {
    "2020-03-09": {
        "stage_name": "Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh số 17 & Phong tỏa)",
        "news_url": "https://cafef.vn/giam-579-trong-sang-9-3-vn-index-ghi-nhan-muc-giam-sau-thu-2-trong-lich-su-ke-tu-nam-2002-20200309121212453.chn",
    },
    "2020-06-11": {
        "stage_name": "Lo ngại làn sóng COVID-19 thứ 2 & Áp lực chốt lời sau sóng hồi",
        "news_url": "https://cafef.vn/chung-khoan-viet-nam-giam-sau-theo-da-do-lua-cua-thi-truong-toan-cau-202006111710123.chn",
    },
    "2020-06-23": {
        "stage_name": "Rung lắc kỹ thuật & Thận trọng trước Báo cáo KQKD Quý 2/2020",
        "news_url": "https://vietstock.vn/2020/06/chung-khoan-dieu-chinh-ngan-han-145-768120.htm",
    },
    "2020-07-13": {
        "stage_name": "Áp lực bán tháo ngắn hạn tại vùng kháng cự",
        "news_url": "https://cafef.vn/ap-luc-ban-thao-xuat-hien-vn-index-dieu-chinh-sau-202007131610221.chn",
    },
    "2020-07-20": {
        "stage_name": "Làn sóng dịch COVID-19 thứ 2 bùng phát tại Đà Nẵng",
        "news_url": "https://vnexpress.net/vn-index-giam-sau-nhat-3-thang-vi-covid-19-o-da-nang-4136691.html",
    },
    "2022-05-09": {
        "stage_name": "Siết tín dụng BĐS, xử lý sai phạm Tân Hoàng Minh/FLC & Call Margin",
        "news_url": "https://cafef.vn/vn-index-giam-ky-luc-ap-luc-call-margin-lan-rong-20220509163012.chn",
    },
    "2022-05-12": {
        "stage_name": "Làn sóng bán giải chấp (Force Sell) & Lo ngại FED tăng lãi suất",
        "news_url": "https://cafef.vn/lan-song-ban-giai-chap-khien-thi-truong-chuyen-dong-tieu-cuc-20220513091512.chn",
    },
    "2022-06-02": {
        "stage_name": "Áp lực lạm phát kỷ lục toàn cầu & Chính sách thắt chặt tiền tệ",
        "news_url": "https://vneconomy.vn/ap-luc-lam-phat-toan-cau-khien-chung-khoan-bi-ban-thao.htm",
    },
    "2022-06-10": {
        "stage_name": "Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB - Vạn Thịnh Phát",
        "news_url": "https://cafef.vn/ba-truong-my-lan-bi-buoc-boi-thuong-cho-scb-gan-674000-ty-dong-188240412063955273.chn",
    },
    "2024-06-14": {
        "stage_name": "Áp lực Tỷ giá USD/VND tăng cao & NHNN phát hành tín phiếu hút tiền",
        "news_url": "https://vneconomy.vn/nhnn-hut-tien-qua-kenh-tin-phieu-ap-luc-ty-gia-tang-nhiet.htm",
    },
    "2025-04-03": {
        "stage_name": "Biến động địa chính trị toàn cầu & Dòng vốn ngoại bán ròng",
        "news_url": "https://cafef.vn/bien-dong-dia-chinh-tri-va-lo-ngai-thue-quan-tac-dong-manh-den-vn-index.chn",
    },
    "2025-05-19": {
        "stage_name": "Chốt lời ngắn hạn & Rung lắc áp lực Margin nhóm Midcap",
        "news_url": "https://vietstock.vn/chot-loi-ngan-han-ap-luc-giai-chap-cuc-bo.htm",
    },
    "2026-03-23": {
        "stage_name": "Căng thẳng địa chính trị Trung Đông & Điều chỉnh vĩ mô đầu năm",
        "news_url": "https://cafef.vn/giua-boi-canh-chien-su-iran-lich-su-chung-khoan-viet-lap-lai-sau-6-nam-dieu-gi-dang-cho-doi-phia-truoc-188260309163948338.chn",
    },
}
# Gán thông tin stage_name và news_url dựa trên start_date
crisis_df["stage_name"] = crisis_df["start_date"].apply(
    lambda x: news_mapping.get(x, {}).get("stage_name", "Điều chỉnh vĩ mô")
)
crisis_df["news_url"] = crisis_df["start_date"].apply(
    lambda x: news_mapping.get(x, {}).get("news_url", "")
)

In [95]:
from pathlib import Path

# Đường dẫn thư mục dự án
PROJECT_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)

# Xuất file CSV hoàn chỉnh
output_csv = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "preparation_preprocessing"
    / "crisis_periods.csv"
)

output_csv.parent.mkdir(parents=True, exist_ok=True)
crisis_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Đã lưu file tại: {output_csv}")

Đã lưu file tại: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\crisis_periods.csv


In [96]:
import pandas as pd
from pathlib import Path
import shutil

# Khai báo đường dẫn thư mục chứa dữ liệu
data_dir = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing"
)

original_path = data_dir / "crisis_periods.csv"
test_path = data_dir / "crisis_periods_test.csv"

# Kiểm tra file gốc
if not original_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {original_path}")

# Sao chép toàn bộ dữ liệu gốc sang file test
shutil.copy2(original_path, test_path)

# Đọc dữ liệu gốc
df = pd.read_csv(original_path)

# Chỉ giữ lại 2 giai đoạn khủng hoảng chính thức
keep_periods = [
    ("2020-03-09", "2020-06-01"),
    ("2022-06-10", "2024-06-12")
]

df["start_date"] = pd.to_datetime(df["start_date"]).dt.strftime("%Y-%m-%d")
df["end_date"] = pd.to_datetime(df["end_date"]).dt.strftime("%Y-%m-%d")

df_filtered = df[
    df.apply(
        lambda row: (row["start_date"], row["end_date"]) in keep_periods,
        axis=1
    )
].copy()

# Kiểm tra đủ 2 giai đoạn trước khi ghi đè
if len(df_filtered) != 2:
    raise ValueError(
        f"Chỉ tìm thấy {len(df_filtered)}/2 giai đoạn. "
        "File gốc chưa bị thay đổi."
    )

# Ghi đè file gốc bằng 2 dòng đã chọn
df_filtered.to_csv(original_path, index=False, encoding="utf-8-sig")

# Hiển thị kết quả
print("Đã sao chép dữ liệu gốc sang:", test_path)
print("Đã cập nhật file:", original_path)
print("\nNội dung crisis_periods.csv:")
display(df_filtered)

Đã sao chép dữ liệu gốc sang: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\crisis_periods_test.csv
Đã cập nhật file: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\crisis_periods.csv

Nội dung crisis_periods.csv:


,start_date,end_date,max_drawdown,stage_name,news_url
0,2020-03-09,2020-06-01,-35.68%,Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ...,https://cafef.vn/giam-579-trong-sang-9-3-vn-in...
8,2022-06-10,2024-06-12,-40.34%,"Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB...",https://cafef.vn/ba-truong-my-lan-bi-buoc-boi-...


2.3. Xác định đỉnh/đáy/ngày phục hồi của từng mã trong mỗi giai đoạn

In [97]:
import pandas as pd
from pathlib import Path


# Đọc dữ liệu
# Đọc hai tập dữ liệu:

BASE_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)

# Đọc hai tập dữ liệu vào hai biến riêng biệt
tickers = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "20tickers_typed.csv"
)

crisis = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "crisis_periods.csv"
)

# Chuyển kiểu dữ liệu ngày: Chuyển các cột thời gian sang định dạng datetime để có thể thực hiện các phép so sánh và tính toán khoảng thời gian. 

tickers["time"] = pd.to_datetime(
    tickers["time"]
)

crisis["start_date"] = pd.to_datetime(
    crisis["start_date"]
)

crisis["end_date"] = pd.to_datetime(
    crisis["end_date"]
)

results = []

# Tính cho từng mã: Lặp qua toàn bộ 20 mã cổ phiếu. 

for ticker in tickers["ticker"].unique():

    stock = (
        tickers[tickers["ticker"] == ticker]
        .sort_values("time")
        .copy()
    )
# Mỗi mã cổ phiếu sẽ được ghép với toàn bộ các sự kiện khủng hoảng. 
    for _, row in crisis.iterrows():

        start_date = row["start_date"]
        end_date = row["end_date"]

        # Peak: đỉnh trong 60 ngày trước khủng hoảng
        #Lấy dữ liệu 60 ngày trước khi khủng hoảng bắt đầu

        pre_crisis = stock[
            (stock["time"] < start_date)
            &
            (
                stock["time"]
                >= start_date - pd.Timedelta(days=60)
            )
        ]

        if pre_crisis.empty:
            continue
        #Tìm giá đóng cửa lớn nhất 
        peak_idx = pre_crisis["close"].idxmax()
       #Lấy giá và ngày đạt đỉnh 
        peak_price = stock.loc[
            peak_idx,
            "close"
        ]

        peak_date = stock.loc[
            peak_idx,
            "time"
        ]

        # Trough: đáy trong khủng hoảng
        #L ấy dữ liệu trong thời gian khủng hoảng: 
        
        crisis_period = stock[
            (stock["time"] >= start_date)
            &
            (stock["time"] <= end_date)
        ]
        #Tìm giá thấp nhất
        if crisis_period.empty:
            continue

        trough_idx = crisis_period["close"].idxmin()
       #Lấy giá và ngày tương ứng: 
        trough_price = stock.loc[
            trough_idx,
            "close"
        ]

        trough_date = stock.loc[
            trough_idx,
            "time"
        ]

        # Recovery
        # Bước 1: Lấy dữ liệu sau khi khủng hoảng kết thúc 
        after_crisis = stock[
            stock["time"] > end_date
        ]
      #Bước 2: Tìm ngày đầu tiên giá quay lại mức Peak 
        recovery = after_crisis[
            after_crisis["close"]
            >= peak_price
        ]

        if not recovery.empty:

            recovery_date = recovery.iloc[0]["time"]
      #Tính số ngày phục hồi 
            recovery_days = (
                recovery_date
                - trough_date
            ).days

            recovery_info = (
                f"Recovered in {recovery_days} days"
            )

        else:
       #Xử lý trường hợp chưa phục hồi 
            recovery_date = pd.NaT

            recovery_days = (
                stock["time"].max()
                - trough_date
            ).days

            recovery_info = (
                f"Not recovered by "
                f"{stock['time'].max().date()} "
                f"({recovery_days} days)"
            )

        results.append({

            "ticker": ticker,
            "stage_name": row["stage_name"],

            "start_date": start_date,
            "end_date": end_date,

            "peak_date": peak_date,
            "peak_price": peak_price,

            "trough_date": trough_date,
            "trough_price": trough_price,

            "recovery_date": recovery_date,
            "recovery_info": recovery_info

        })

# Kết quả

events = pd.DataFrame(results)

print("\n===== 5 DÒNG ĐẦU =====")
print(events.head())

print("\n===== TỔNG SỐ SỰ KIỆN =====")
print(len(events))

not_recovered = events[
    events["recovery_date"].isna()
]

print("\n===== CHƯA PHỤC HỒI =====")

print(
    not_recovered[
        [
            "ticker",
            "stage_name",
            "peak_date",
            "peak_price",
            "trough_date",
            "trough_price",
            "recovery_info"
        ]
    ]
)

print(
    "\nSố trường hợp chưa phục hồi:",
    len(not_recovered)
)


===== 5 DÒNG ĐẦU =====
  ticker                                         stage_name start_date  \
0    BID  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
1    BID  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
2    CTG  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
3    CTG  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
4    GAS  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   

    end_date  peak_date  peak_price trough_date  trough_price recovery_date  \
0 2020-06-01 2020-01-22       28.99  2020-03-23         16.09    2022-01-13   
1 2024-06-12 2022-04-13       27.45  2022-10-11         19.13    2024-06-13   
2 2020-06-01 2020-02-06       12.51  2020-03-31          7.75    2020-10-12   
3 2024-06-12 2022-04-13       19.04  2022-10-11         11.97    2024-06-13   
4 2020-06-01 2020-01-10       57.76  2020-03-23         32.46    2021-01-07   

           recovery_info  
0  Recovered in 661 days  
1 

In [98]:
# Lưu kết quả

output_path = BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "crisis_stock_events.csv"

events.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Đã lưu file tại:", output_path)
print("File tồn tại:", output_path.exists())

Đã lưu file tại: G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu\data\processed\preparation_preprocessing\crisis_stock_events.csv
File tồn tại: True


3. Xây dựng biến mục tiêu và đặc trưng

3.1. Tính toán biến đặc trưng trong và sau khủng hoảng

In [104]:
import pandas as pd
from pathlib import Path

# ĐỌC DỮ LIỆU ĐẦU VÀO

# - 20tickers_typed.csv: dữ liệu giá cổ phiếu VN30
# - vnindex_typed.csv: dữ liệu VN-Index
# - crisis_stock_events.csv
#   (đã có peak, trough, recovery)

BASE_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)

stocks = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "20tickers_typed.csv"
)

vnindex = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "vnindex_typed.csv"
)

events = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "crisis_stock_events.csv"
)

# CHUẨN HÓA KIỂU DỮ LIỆU NGÀY

# Chuyển toàn bộ cột thời gian sang datetime
# để có thể lọc dữ liệu theo khoảng thời gian
# và thực hiện các phép tính thời gian chính xác.

stocks["time"] = pd.to_datetime(stocks["time"])
vnindex["time"] = pd.to_datetime(vnindex["time"])

events["start_date"] = pd.to_datetime(events["start_date"])
events["end_date"] = pd.to_datetime(events["end_date"])
events["peak_date"] = pd.to_datetime(events["peak_date"])
events["trough_date"] = pd.to_datetime(events["trough_date"])

results = []

# TÍNH CÁC CHỈ SỐ CHO TỪNG SỰ KIỆN
# Với mỗi dòng trong crisis_stock_events:
#   - Resilience
#   - Relative Resilience
#   - Recovery Ratio 60 phiên

for _, row in events.iterrows():

    ticker = row["ticker"]

    start_date = row["start_date"]
    end_date = row["end_date"]

    peak_price = row["peak_price"]
    trough_price = row["trough_price"]

    trough_date = row["trough_date"]

    # RESILIENCE
    # Đo mức sụt giảm của cổ phiếu trong khủng hoảng.
    # Công thức:
    # Resilience = (Trough - Peak) / Peak
    # Giá trị:
    # - Gần 0: giảm ít, chống chịu tốt
    # - Âm lớn: giảm sâu, chống chịu kém

    resilience = (
        trough_price - peak_price
    ) / peak_price

    # RESILIENCE CỦA VN-INDEX
    # Tính theo đúng giai đoạn khủng hoảng tương ứng.
    # Peak VN-Index:
    #   đỉnh trong 60 ngày trước khủng hoảng
    # Trough VN-Index:
    #   đáy trong thời gian khủng hoảng

    vn_pre = vnindex[
        (vnindex["time"] < start_date)
        &
        (
            vnindex["time"]
            >= start_date - pd.Timedelta(days=60)
        )
    ]

    vn_crisis = vnindex[
        (vnindex["time"] >= start_date)
        &
        (vnindex["time"] <= end_date)
    ]

    vn_peak = vn_pre["close"].max()
    vn_trough = vn_crisis["close"].min()

    vn_resilience = (
        vn_trough - vn_peak
    ) / vn_peak

    # RELATIVE RESILIENCE
    # So sánh khả năng chống chịu của cổ phiếu
    # với toàn thị trường.
    # Công thức:
    # Relative Resilience
    # = Resilience_stock - Resilience_vnindex
    # Ý nghĩa:
    # > 0 : tốt hơn thị trường
    # < 0 : kém hơn thị trường

    relative_resilience = (
        resilience - vn_resilience
    )

    # RECOVERY RATIO 60 PHIÊN
    # Đo mức độ phục hồi sau 60 phiên giao dịch kể từ ngày tạo đáy.
    # Công thức:
    # (Price_60d - Trough) / (Peak - Trough)
    # Ý nghĩa:
    # = 1   : hồi hoàn toàn về đỉnh cũ
    # > 1   : vượt đỉnh cũ
    # = 0.5 : hồi được 50% quãng đường
    # < 0   : còn giảm tiếp

    stock = (
        stocks[stocks["ticker"] == ticker]
        .sort_values("time")
        .reset_index(drop=True)
    )

    trough_rows = stock[
        stock["time"] == trough_date
    ]

    # Kiểm tra có tìm được ngày đáy hay không

    if len(trough_rows) == 0:

        recovery_ratio_60d = None
        note = "Không tìm thấy ngày đáy"

    else:

        trough_idx = trough_rows.index[0]

        # Lấy phiên giao dịch thứ 60 sau đáy

        target_idx = trough_idx + 60

        if target_idx < len(stock):

            price_60d = stock.loc[
                target_idx,
                "close"
            ]

            recovery_ratio_60d = (
                price_60d - trough_price
            ) / (
                peak_price - trough_price
            )

            note = "Đủ dữ liệu"

        else:

            # Không đủ 60 phiên sau đáy
            # Thường xảy ra với các sự kiện
            # gần cuối bộ dữ liệu năm 2026.

            recovery_ratio_60d = None
            note = "Chưa đủ dữ liệu"

    # LƯU KẾT QUẢ

    results.append({

        **row.to_dict(),

        "resilience": resilience,

        "relative_resilience":
            relative_resilience,

        "recovery_ratio_60d":
            recovery_ratio_60d,

        "recovery_ratio_note":
            note
    })

# TẠO DATAFRAME KẾT QUẢ
# Bổ sung thêm:
#   - resilience
#   - relative_resilience
#   - recovery_ratio_60d
#   - recovery_ratio_note

crisis_label = pd.DataFrame(results)

# LƯU FILE OUTPUT
# File đầu ra:
# data/processed/crisis_label.csv
# Đây là dataset cuối cùng dùng cho bước
# xây dựng nhãn và mô hình học máy.

output_path = (
    BASE_DIR
    / "data"
    / "processed"
    / "preparation_preprocessing"
    / "crisis_label.csv"
)

crisis_label.to_csv(
    output_path,
    index=False
)

# KIỂM TRA KẾT QUẢ

print("\n===== 5 DÒNG ĐẦU =====")
print(crisis_label.head())

print("\n===== KÍCH THƯỚC =====")
print(crisis_label.shape)

print("\n===== THIẾU DỮ LIỆU 60 PHIÊN =====")

print(
    crisis_label[
        crisis_label[
            "recovery_ratio_note"
        ] == "Chưa đủ dữ liệu"
    ][
        [
            "ticker",
            "stage_name",
            "trough_date"
        ]
    ]
)


===== 5 DÒNG ĐẦU =====
  ticker                                         stage_name start_date  \
0    BID  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
1    BID  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
2    CTG  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
3    CTG  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
4    GAS  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   

    end_date  peak_date  peak_price trough_date  trough_price recovery_date  \
0 2020-06-01 2020-01-22       28.99  2020-03-23         16.09    2022-01-13   
1 2024-06-12 2022-04-13       27.45  2022-10-11         19.13    2024-06-13   
2 2020-06-01 2020-02-06       12.51  2020-03-31          7.75    2020-10-12   
3 2024-06-12 2022-04-13       19.04  2022-10-11         11.97    2024-06-13   
4 2020-06-01 2020-01-10       57.76  2020-03-23         32.46    2021-01-07   

           recovery_info  resilience  relative_resilienc

3.2. Tính các đặc trưng kỹ thuật của từng mã trước khủng hoảng

In [105]:
import pandas as pd
from pathlib import Path

# TẠO ĐẶC TRƯNG TRƯỚC KHỦNG HOẢNG
# Với mỗi cặp (ticker, stage_name):
# - Volatility 60 ngày trước khủng hoảng
# - Mean Return 60 ngày trước khủng hoảng
# - Momentum 60 ngày trước khủng hoảng
# - Average Volume 60 ngày trước khủng hoảng
# Đồng thời tìm các sự kiện khủng hoảng xảy ra cách nhau <= 60 ngày để ghi nhận khả năng chồng lấn cửa sổ dữ liệu.

# XÁC ĐỊNH THƯ MỤC GỐC DỰ ÁN

BASE_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)

# ĐỌC DỮ LIỆU GIÁ CỔ PHIẾU
# 20tickers_typed.csv: chứa dữ liệu giá và khối lượng giao dịch của 20 cổ phiếu VN30

tickers = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "20tickers_typed.csv"
)

# ĐỌC DANH SÁCH CÁC SỰ KIỆN KHỦNG HOẢNG
# crisis_stock_events.csv: chứa các cặp (mã cổ phiếu, sự kiện) đã được xác định ở bước trước

events = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "crisis_stock_events.csv"
)

# CHUYỂN CÁC CỘT NGÀY THÀNH DATETIME
# Giúp thực hiện:
# - lọc dữ liệu theo ngày
# - tính khoảng cách thời gian
# - sắp xếp theo thời gian

tickers["time"] = pd.to_datetime(
    tickers["time"]
)

events["start_date"] = pd.to_datetime(
    events["start_date"]
)

events["end_date"] = pd.to_datetime(
    events["end_date"]
)

# SẮP XẾP DỮ LIỆU THEO:
# ticker -> thời gian
# Việc sắp xếp giúp tính return chính xác

tickers = tickers.sort_values(
    ["ticker", "time"]
)

# TÍNH DAILY RETURN
# Công thức:
# Return_t = (Price_t - Price_t-1) / Price_t-1
# Hoặc:
# Return_t = Price_t / Price_t-1 - 1
# Daily return sẽ được dùng để tính:
# - Volatility
# - Mean Return

tickers["daily_return"] = (
    tickers
    .groupby("ticker")["close"]
    .pct_change()
)

# DANH SÁCH LƯU KẾT QUẢ

feature_rows = []

# DUYỆT TỪNG DÒNG TRONG
# crisis_stock_events.csv
# Mỗi dòng tương ứng 1 mã cổ phiếu + 1 sự kiện khủng hoảng

for _, row in events.iterrows():

    ticker = row["ticker"]
    stage_name = row["stage_name"]
    start_date = row["start_date"]

    # LỌC RIÊNG DỮ LIỆU CỦA MÃ ĐANG XÉT

    stock = (
        tickers[
            tickers["ticker"] == ticker
        ]
        .sort_values("time")
        .copy()
    )

    # LẤY 60 PHIÊN GIAO DỊCH TRƯỚC KHỦNG HOẢNG
    # CHỈ DÙNG DỮ LIỆU TRƯỚC start_date
    # KHÔNG sử dụng dữ liệu:
    # - trong khủng hoảng
    # - sau khủng hoảng
    # Tránh Data Leakage

    pre_crisis = stock[
        stock["time"] < start_date
    ].tail(60)

    # KIỂM TRA ĐỦ 60 PHIÊN KHÔNG

    if len(pre_crisis) < 60:
        volatility = None
        mean_return = None
        momentum = None
        avg_volume = None
        note = "Không đủ 60 phiên"

    else:

        # VOLATILITY
        # Độ lệch chuẩn của daily return
        # Đo mức độ biến động giá
        # Giá trị càng lớn:
        # biến động càng mạnh

        volatility = (
            pre_crisis["daily_return"]
            .std()
        )

        # RETURN TRUNG BÌNH
        # Trung bình daily return
        # Dương: xu hướng tăng
        # Âm: xu hướng giảm
        
        mean_return = (
            pre_crisis["daily_return"]
            .mean()
        )

        # MOMENTUM
        # Công thức: (Giá cuối - Giá đầu)/Giá đầu
        # Đo xu hướng giá trong
        # toàn bộ 60 phiên
        
        first_price = (
            pre_crisis.iloc[0]["close"]
        )

        last_price = (
            pre_crisis.iloc[-1]["close"]
        )

        momentum = (
            last_price - first_price
        ) / first_price

        # KHỐI LƯỢNG GIAO DỊCH TRUNG BÌNH
        # Đo thanh khoản của cổ phiếu trong 60 phiên trước khủng hoảng

        avg_volume = (
            pre_crisis["volume"]
            .mean()
        )

        note = "Đủ dữ liệu"

    # LƯU KẾT QUẢ

    feature_rows.append({
        "ticker": ticker,
        "stage_name": stage_name,
        "start_date": start_date,
        "volatility_60d": volatility,
        "mean_return_60d": mean_return,
        "momentum_60d": momentum,
        "avg_volume_60d": avg_volume,
        "note": note
    })

# CHUYỂN KẾT QUẢ THÀNH DATAFRAME

features_df = pd.DataFrame(
    feature_rows
)

# LƯU FILE ĐẶC TRƯNG
# File này sẽ được sử dụng cho bước Machine Learning

output_path = (
    BASE_DIR
    / "data"
    / "processed"
    / "preparation_preprocessing"
    / "features_pre_crisis.csv"
)

features_df.to_csv(
    output_path,
    index=False
)

# HIỂN THỊ KẾT QUẢ

print("\n===== 5 DÒNG ĐẦU =====")

print(
    features_df.head()
)

print("\n===== KÍCH THƯỚC =====")

print(
    features_df.shape
)

# TÌM CÁC SỰ KIỆN KHỦNG HOẢNG
# CÁCH NHAU <= 60 NGÀY
# Mục đích: phát hiện khả năng chồng lấn cửa sổ dữ liệu trước khủng hoảng

crisis_dates = (
    events[
        [
            "stage_name",
            "start_date"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "start_date"
    )
)

close_events = []
for i in range(
    1,
    len(crisis_dates)
):
    previous_event = (
        crisis_dates.iloc[i - 1]
    )
    current_event = (
        crisis_dates.iloc[i]
    )
    gap_days = (
        current_event["start_date"]
        -
        previous_event["start_date"]
    ).days

    if gap_days <= 60:
        close_events.append({
            "event_1":
                previous_event["stage_name"],
            "date_1":
                previous_event["start_date"],
            "event_2":
                current_event["stage_name"],
            "date_2":
                current_event["start_date"],
            "gap_days":
                gap_days
        })

close_df = pd.DataFrame(
    close_events
)

# LƯU DANH SÁCH CÁC SỰ KIỆN GẦN NHAU

close_df.to_csv(
    BASE_DIR
    / "data"
    / "processed"
    / "preparation_preprocessing"
    / "close_crisis_events.csv",

    index=False
)

# HIỂN THỊ

print(
    "\n===== CÁC SỰ KIỆN CÁCH NHAU <= 60 NGÀY ====="
)

print(close_df)

print(
    "\nSố cặp sự kiện gần nhau:",
    len(close_df)
)

print(
    "\nĐã lưu features_pre_crisis.csv"
)

print(
    "Đã lưu close_crisis_events.csv"
)


===== 5 DÒNG ĐẦU =====
  ticker                                         stage_name start_date  \
0    BID  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
1    BID  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
2    CTG  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
3    CTG  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
4    GAS  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   

   volatility_60d  mean_return_60d  momentum_60d  avg_volume_60d        note  
0        0.026105         0.002975      0.181555    9.012377e+05  Đủ dữ liệu  
1        0.026535        -0.002642     -0.180531    1.848185e+06  Đủ dữ liệu  
2        0.024457         0.004461      0.271038    6.123410e+06  Đủ dữ liệu  
3        0.026510        -0.002003     -0.133092    4.920037e+06  Đủ dữ liệu  
4        0.018298        -0.003736     -0.209082    2.663432e+05  Đủ dữ liệu  

===== KÍCH THƯỚC =====
(40, 8)

===== CÁC SỰ KIỆN CÁCH N

3.3. Tính đặc trưng Beta thị trường

In [106]:
import pandas as pd
import numpy as np
from pathlib import Path

# TASK 8: TÍNH ĐẶC TRƯNG BETA THỊ TRƯỜNG

BASE_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)
PROCESSED = BASE_DIR / "data" / "processed" / "preparation_preprocessing"

WINDOW = 120        # số phiên dùng để tính beta
MAX_GAP_DAYS = 120  # ngưỡng "sát nhau" để ghi chú

# ĐỌC DỮ LIỆU
tickers = pd.read_csv(PROCESSED / "20tickers_typed.csv")
vnindex = pd.read_csv(PROCESSED / "vnindex_typed.csv")
events = pd.read_csv(PROCESSED / "crisis_stock_events.csv")
features = pd.read_csv(PROCESSED / "features_pre_crisis.csv")

tickers["time"] = pd.to_datetime(tickers["time"])
vnindex["time"] = pd.to_datetime(vnindex["time"])
events["start_date"] = pd.to_datetime(events["start_date"])
features["start_date"] = pd.to_datetime(features["start_date"])

# TÍNH DAILY RETURN
tickers = tickers.sort_values(["ticker", "time"]).reset_index(drop=True)
tickers["ret"] = tickers.groupby("ticker")["close"].pct_change()

vnindex = vnindex.sort_values("time").reset_index(drop=True)
vnindex["ret_mkt"] = vnindex["close"].pct_change()
vnindex = vnindex[["time", "ret_mkt"]]

# GHÉP RETURN CỔ PHIẾU VỚI RETURN VN-INDEX THEO NGÀY
merged = tickers[["time", "ticker", "ret"]].merge(vnindex, on="time", how="inner")
merged = merged.dropna(subset=["ret", "ret_mkt"])

# TÍNH BETA CHO TỪNG (ticker, stage_name) - chỉ dùng dữ liệu < start_date
beta_rows = []

for _, row in events.iterrows():
    ticker = row["ticker"]
    start_date = row["start_date"]

    window = (
        merged[(merged["ticker"] == ticker) & (merged["time"] < start_date)]
        .sort_values("time")
        .tail(WINDOW)
    )

    if len(window) < WINDOW:
        beta = np.nan
        note = f"Không đủ {WINDOW} phiên"
    else:
        x = window["ret_mkt"].values
        y = window["ret"].values
        beta = np.polyfit(x, y, 1)[0]
        note = "Đủ dữ liệu"

    beta_rows.append({
        "ticker": ticker,
        "stage_name": row["stage_name"],
        "start_date": start_date,
        "beta": beta,
        "beta_note": note,
    })

beta_df = pd.DataFrame(beta_rows)

# BỔ SUNG CỘT BETA VÀO BẢNG ĐẶC TRƯNG
out = features.merge(
    beta_df,
    on=["ticker", "stage_name", "start_date"],
    how="left",
    validate="one_to_one",
)
out.to_csv(PROCESSED / "beta_pre_crisis.csv", index=False)

print("===== 5 DÒNG ĐẦU =====")
print(out.head())
print("\n===== KÍCH THƯỚC =====")
print(out.shape)
print("\n===== SỐ DÒNG THIẾU BETA =====")
print(out["beta"].isna().sum())
print("\n===== THỐNG KÊ BETA =====")
print(out["beta"].describe())

# CÁC GIAI ĐOẠN KHỦNG HOẢNG CÁCH NHAU <= 120 NGÀY
crisis_dates = (
    events[["stage_name", "start_date"]]
    .drop_duplicates()
    .sort_values("start_date")
    .reset_index(drop=True)
)

close_events = []
for i in range(1, len(crisis_dates)):
    prev_e = crisis_dates.iloc[i - 1]
    cur_e = crisis_dates.iloc[i]
    gap_days = (cur_e["start_date"] - prev_e["start_date"]).days
    if gap_days <= MAX_GAP_DAYS:
        close_events.append({
            "event_1": prev_e["stage_name"],
            "date_1": prev_e["start_date"].date(),
            "event_2": cur_e["stage_name"],
            "date_2": cur_e["start_date"].date(),
            "gap_days": gap_days,
        })

close_df = pd.DataFrame(close_events)
close_df.to_csv(PROCESSED / "close_crisis_events_120d.csv", index=False)

print(f"\n===== CÁC SỰ KIỆN CÁCH NHAU <= {MAX_GAP_DAYS} NGÀY =====")
print(close_df)
print("\nSố cặp sự kiện gần nhau:", len(close_df))
print("Đã lưu beta_pre_crisis.csv và close_crisis_events_120d.csv")

===== 5 DÒNG ĐẦU =====
  ticker                                         stage_name start_date  \
0    BID  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
1    BID  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
2    CTG  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   
3    CTG  Khủng hoảng Trái phiếu doanh nghiệp, Sự cố SCB... 2022-06-10   
4    GAS  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09   

   volatility_60d  mean_return_60d  momentum_60d  avg_volume_60d        note  \
0        0.026105         0.002975      0.181555    9.012377e+05  Đủ dữ liệu   
1        0.026535        -0.002642     -0.180531    1.848185e+06  Đủ dữ liệu   
2        0.024457         0.004461      0.271038    6.123410e+06  Đủ dữ liệu   
3        0.026510        -0.002003     -0.133092    4.920037e+06  Đủ dữ liệu   
4        0.018298        -0.003736     -0.209082    2.663432e+05  Đủ dữ liệu   

       beta   beta_note  
0  1.761554  Đủ dữ liệu  

3.4. Gán đặc trưng ngành

In [107]:
import pandas as pd
from pathlib import Path

# TASK 8.2: GÁN ĐẶC TRƯNG NGÀNH

BASE_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)

# Đọc hai tập dữ liệu vào hai biến riêng biệt
tickers = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "20tickers_typed.csv"
)

crisis = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "crisis_periods.csv"
)
PROCESSED = BASE_DIR / "data" / "processed" / "preparation_preprocessing"

# BẢNG PHÂN LOẠI 20 MÃ VÀO 4 NHÓM NGÀNH
sector_map = {
    "Banking":    ["VCB", "CTG", "BID", "STB", "MBB", "TCB", "VPB", "HDB"],
    "RealEstate": ["VIC", "VHM", "VRE", "NVL", "PDR"],
    "Energy":     ["GAS", "PLX", "PVD"],
    "Consumer":   ["VNM", "MWG", "MSN", "SAB"],
}

rows = [{"ticker": tk, "sector": sec}
        for sec, tks in sector_map.items() for tk in tks]
branches = pd.DataFrame(rows)
assert len(branches) == 20 and branches["ticker"].is_unique

branches.to_csv(PROCESSED / "branches_division.csv", index=False)
print(branches)
print(branches["sector"].value_counts())


# GẮN NGÀNH VÀO BẢNG ĐẶC TRƯNG (đã có beta từ Task 8)
features = pd.read_csv(PROCESSED / "beta_pre_crisis.csv")
n_before = len(features)

details = features.merge(branches, on="ticker", how="left", validate="many_to_one")
assert len(details) == n_before
missing = details.loc[details["sector"].isna(), "ticker"].unique()
print("Mã chưa có ngành:", list(missing))
assert len(missing) == 0

# ONE-HOT ENCODING
sector_cols = ["sector_Banking", "sector_Consumer", "sector_Energy", "sector_RealEstate"]
for col in sector_cols:
    details[col] = (details["sector"] == col.replace("sector_", "")).astype(int)

assert (details[sector_cols].sum(axis=1) == 1).all()
assert (details.groupby("ticker")["sector"].nunique() == 1).all()

details = details.drop(columns=["sector"])
details.to_csv(PROCESSED / "details_pre_crisis.csv", index=False)

print(details.shape)
print(details[sector_cols].sum())

   ticker      sector
0     VCB     Banking
1     CTG     Banking
2     BID     Banking
3     STB     Banking
4     MBB     Banking
5     TCB     Banking
6     VPB     Banking
7     HDB     Banking
8     VIC  RealEstate
9     VHM  RealEstate
10    VRE  RealEstate
11    NVL  RealEstate
12    PDR  RealEstate
13    GAS      Energy
14    PLX      Energy
15    PVD      Energy
16    VNM    Consumer
17    MWG    Consumer
18    MSN    Consumer
19    SAB    Consumer
sector
Banking       8
RealEstate    5
Consumer      4
Energy        3
Name: count, dtype: int64
Mã chưa có ngành: []
(40, 14)
sector_Banking       16
sector_Consumer       8
sector_Energy         6
sector_RealEstate    10
dtype: int64


3.5. Ghép bảng đặc trưng + nhãn thành dataset hoàn chỉnh

In [108]:
import pandas as pd
import numpy as np
from pathlib import Path

# TASK 8.3: GHÉP ĐẶC TRƯNG + NHÃN THÀNH df_model_train.csv

BASE_DIR = Path(
    r"G:\My Drive\BÀI TẬP NHÓM\Năm 3\[Nhóm 10] ML & AI trong tài chính\ML_ChongChiu&PhucHoiCoPhieu"
)

# Đọc hai tập dữ liệu vào hai biến riêng biệt
tickers = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "20tickers_typed.csv"
)

crisis = pd.read_csv(
    BASE_DIR / "data" / "processed" / "preparation_preprocessing" / "crisis_periods.csv"
)
PROCESSED = BASE_DIR / "data" / "processed" / "preparation_preprocessing"

# ĐỌC HAI BẢNG
key = ["ticker", "stage_name", "start_date"]
label_cols = ["resilience", "relative_resilience", "recovery_ratio_60d"]

details = pd.read_csv(PROCESSED / "details_pre_crisis.csv")
details["start_date"] = pd.to_datetime(details["start_date"])

def compute_labels():
    """Tính 3 nhãn theo đúng công thức script 7.1.2, không ghi file."""
    px = pd.read_csv(PROCESSED / "20tickers_typed.csv")
    vn = pd.read_csv(PROCESSED / "vnindex_typed.csv")
    ev = pd.read_csv(PROCESSED / "crisis_stock_events.csv")
    px["time"] = pd.to_datetime(px["time"])
    vn["time"] = pd.to_datetime(vn["time"])
    for c in ["start_date", "end_date", "peak_date", "trough_date"]:
        ev[c] = pd.to_datetime(ev[c])

    out = []
    for _, r in ev.iterrows():
        # Resilience = (Trough - Peak) / Peak
        res = (r["trough_price"] - r["peak_price"]) / r["peak_price"]

        # Resilience của VN-Index cùng giai đoạn
        vn_pre = vn[(vn["time"] < r["start_date"]) &
                    (vn["time"] >= r["start_date"] - pd.Timedelta(days=60))]
        vn_cr = vn[(vn["time"] >= r["start_date"]) & (vn["time"] <= r["end_date"])]
        vn_res = (vn_cr["close"].min() - vn_pre["close"].max()) / vn_pre["close"].max()

        # Recovery ratio 60 phiên sau đáy
        s = px[px["ticker"] == r["ticker"]].sort_values("time").reset_index(drop=True)
        tr = s[s["time"] == r["trough_date"]]
        ratio = np.nan
        if len(tr) and tr.index[0] + 60 < len(s):
            p60 = s.loc[tr.index[0] + 60, "close"]
            ratio = (p60 - r["trough_price"]) / (r["peak_price"] - r["trough_price"])

        out.append({
            "ticker": r["ticker"],
            "stage_name": r["stage_name"],
            "start_date": r["start_date"],
            "resilience": res,
            "relative_resilience": res - vn_res,
            "recovery_ratio_60d": ratio,
        })
    return pd.DataFrame(out)

label = pd.read_csv(PROCESSED / "crisis_label.csv")
label["start_date"] = pd.to_datetime(label["start_date"])

if all(c in label.columns for c in label_cols):
    print("Dùng nhãn có sẵn trong crisis_label.csv")
else:
    print("crisis_label.csv thiếu cột nhãn -> tính nhãn trực tiếp trong bộ nhớ")
    label = compute_labels()

print("Khóa trùng ở label:", label.duplicated(key).sum())
df = details.merge(label[key + label_cols], on=key, how="inner", validate="one_to_one")
print("Sau merge:", df.shape)
assert len(df) == len(details), f"Merge mất dòng: {len(details)} -> {len(df)}"

# CHỌN CỘT, KIỂM TRA VÀ XÓA DÒNG THIẾU
feature_cols = ["volatility_60d", "mean_return_60d", "momentum_60d",
                "avg_volume_60d", "beta",
                "sector_Banking", "sector_Consumer",
                "sector_Energy", "sector_RealEstate"]
model_cols = key + feature_cols + label_cols
df = df[model_cols].copy()

df[feature_cols + label_cols] = df[feature_cols + label_cols].replace([np.inf, -np.inf], np.nan)

print("===== SỐ GIÁ TRỊ THIẾU THEO CỘT =====")
print(df[feature_cols + label_cols].isna().sum())

n_before = len(df)
df = df.dropna(subset=feature_cols + label_cols).reset_index(drop=True)
print("Số dòng bị xóa:", n_before - len(df), "| còn lại:", len(df))
assert df.duplicated(key).sum() == 0, "Có khóa trùng trong dataset cuối"
print("Số mã:", df["ticker"].nunique(), "| số giai đoạn:", df["stage_name"].nunique())

df.to_csv(PROCESSED / "df_model_train.csv", index=False)
print(df.shape)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
print(df.head())

Dùng nhãn có sẵn trong crisis_label.csv
Khóa trùng ở label: 0
Sau merge: (40, 17)
===== SỐ GIÁ TRỊ THIẾU THEO CỘT =====
volatility_60d         0
mean_return_60d        0
momentum_60d           0
avg_volume_60d         0
beta                   0
sector_Banking         0
sector_Consumer        0
sector_Energy          0
sector_RealEstate      0
resilience             0
relative_resilience    0
recovery_ratio_60d     0
dtype: int64
Số dòng bị xóa: 0 | còn lại: 40
Số mã: 20 | số giai đoạn: 2
(40, 15)
  ticker                                         stage_name start_date  volatility_60d  mean_return_60d  momentum_60d  avg_volume_60d      beta  sector_Banking  sector_Consumer  sector_Energy  sector_RealEstate  resilience  relative_resilience  \
0    BID  Bùng phát đại dịch COVID-19 toàn cầu (Ca bệnh ... 2020-03-09        0.026105         0.002975      0.181555    9.012377e+05  1.761554               1                0              0                  0   -0.444981            -0.109869   
1   